In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint

df = pd.read_csv('coffee_dataset_G17.csv', index_col=0, parse_dates=True)
anchor = "KC=F"
stocks = [c for c in df.columns if c not in [anchor, 'DBA']]

On fait le Test d'Engle-Granger

In [16]:
results = []
for s in stocks:
    temp_pair = df[[anchor, s]].dropna()
    if len(temp_pair) > 1000:
        score, pvalue, _ = coint(temp_pair[anchor], temp_pair[s])
        results.append((s, pvalue))
        print(f"ISN tested: {anchor} / {s} | p-value: {pvalue:.4f}")

results.sort(key=lambda x: x[1])
best_stock = results[0][0]
print(f"Paire: {anchor} / {best_stock} | p-value: {results[0][1]:.4f}")


ISN tested: KC=F / BROS | p-value: 0.0011
ISN tested: KC=F / FARM | p-value: 0.6335
ISN tested: KC=F / KDP | p-value: 0.7552
ISN tested: KC=F / MDLZ | p-value: 0.7006
ISN tested: KC=F / NSRGY | p-value: 0.4602
ISN tested: KC=F / SBUX | p-value: 0.7902
ISN tested: KC=F / SJM | p-value: 0.2097
ISN tested: KC=F / WEST | p-value: 0.0613
Paire: KC=F / BROS | p-value: 0.0011


In [17]:
def run_backtest(df, s1, s2, lookback, vol_threshold, stop_loss_pct):
    spread = df[s1] / df[s2]
    
    mean = spread.rolling(window=lookback).mean()
    std = spread.rolling(window=lookback).std()
    zscore = (spread - mean) / std
    
    recent_vol = spread.pct_change().rolling(20).std()
    long_term_vol = recent_vol.rolling(100).mean()
    is_stable = recent_vol < (long_term_vol * vol_threshold)
    
    position = 0
    daily_returns = [0] * len(df)
    
    for i in range(lookback, len(df)):
        # entree en position si le marché est stable 
        if position == 0 and is_stable.iloc[i]:
            if zscore.iloc[i] > 2.0: position = -1 #spread trop cher ->vvente
            elif zscore.iloc[i] < -2.0: position = 1 # spread trop bas -> achat
            
        elif position != 0:
            ret = position * spread.pct_change().iloc[i]
            
#stoploss
            if (position == 1 and zscore.iloc[i] >= 0) or (position == -1 and zscore.iloc[i] <= 0):
                position = 0
            elif ret < -stop_loss_pct:
                position = 0 #securite
            else:
                daily_returns[i] = ret
                
    return pd.Series(daily_returns, index=df.index)